In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

from math import log, sqrt
from time import time
from pprint import pprint

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as AUC, log_loss, accuracy_score as accuracy
from sklearn.metrics import (mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2,
                             explained_variance_score as EVS)
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler

from keras.models import Sequential
from keras.layers.core import Dense, Dropout
from keras.layers.normalization import BatchNormalization as BatchNorm
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers.advanced_activations import *
from keras.models import load_model

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

/home/bulent/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:34: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
with open('stations-6to31.pkl', 'rb') as f:
    datas = pickle.load(f)
    
x_train6_ = datas['x_train6']
y_train6 = datas['y_train6']
x_train_ = datas['x_train']
y_train = datas['y_train']
x_dev_ = datas['x_dev']
y_dev = datas['y_dev']
x_test_ = datas['x_test']
y_test = datas['y_test']

print(f'x_train shape: {x_train_.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


From the best 21 configurations modes of respective categories are as follows.

**Initializer:** normal

**Layers:** 2

**Batch Size:** 64

**Optimizer:** adamax

**Shuffle:** True

**Scaler:** RobustScaler

**Loss:** mean_absolute_error

In [3]:
def scale_data(scaler, datas):
    # scaler is a scaling function from sklearn library
    # datas is a dictionary, containing 3 sets of x_data with keys - x_train, x_dev, x_test
    # fit on x_train and return the transformed sets of data
    
    x_train = scaler.fit_transform(datas['x_train'].astype(float))
    x_dev = scaler.transform(datas['x_dev'].astype(float))
    x_test = scaler.transform(datas['x_test'].astype(float))
    
    transformed = {'x_train': x_train, 'x_dev': x_dev, 'x_test': x_test}
    return transformed

data6_ = {'x_train': x_train6_, 'x_dev': x_dev_, 'x_test': x_test_}
data_ = {'x_train': x_train_, 'x_dev': x_dev_, 'x_test': x_test_}

data6 = scale_data(RobustScaler(), data6_)
data = scale_data(RobustScaler(), data_)

x_train6 = data6['x_train']
x_train = data['x_train']

x_dev6 = data6['x_dev']
x_dev = data['x_dev']

x_test6 = data6['x_test']
x_test = data['x_test']

print(f'x_train shape: {x_train.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


In [4]:
def radstimator(h1=20, h2=15, num_vars=5):
    init = 'normal'
    
    model = Sequential()
    model.add( Dense( h1, kernel_initializer=init, input_dim=num_vars ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( BatchNorm())
    model.add( Dense( h2, kernel_initializer=init ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( Dropout( rate=0.4 ))
    
    model.add( Dense( 1, kernel_initializer=init, activation='linear' ))
    
    return model

In [9]:
print(x_train6_[:5]) # 'Latitude', 'BSH', 'Temperature(avg)', 'Daylength', 'H0'

[[40.141       5.9         4.22916667  9.19499685 13.69287119]
 [40.141       1.3         7.6375      9.20626964 13.74607822]
 [40.141       0.          6.2375      9.21860396 13.80432176]
 [40.141       0.          3.32916667  9.23198817 13.86758193]
 [40.141       0.7         5.06956522  9.24640976 13.93583675]]


In [5]:
data6_3v_ = {'x_train': x_train6_[:, [1, 3, 4, 0]], 'x_dev': x_dev_[:, [1, 3, 4, 0]], 'x_test': x_test_[:, [1, 3, 4, 0]]}
data_3v_ = {'x_train': x_train_[:, [1, 3, 4, 0]], 'x_dev': x_dev_[:, [1, 3, 4, 0]], 'x_test': x_test_[:, [1, 3, 4, 0]]}

data6_3v = scale_data(RobustScaler(), data6_3v_)
data_3v = scale_data(RobustScaler(), data_3v_)

x_train6_3v = data6_3v['x_train']
x_train_3v = data_3v['x_train']

x_dev6_3v = data6_3v['x_dev']
x_dev_3v = data_3v['x_dev']

x_test6_3v = data6_3v['x_test']
x_test_3v = data_3v['x_test']

print(f'x_train shape: {x_train_3v.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_3v.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_3v.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_3v.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 4), y_train shape: (52416,)
x_train6 shape: (10654, 4), y_train6 shape: (10654,)
x_dev shape: (9011, 4), y_dev shape: (9011,)
x_test shape: (16899, 4), y_test shape: (16899,)


In [6]:

validation_data6 = ( x_dev6_3v, y_dev )
for i in range(50):
    rads = radstimator(20, 15, 4)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-3vars-h/6stations-h-nNPhiH0 {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train6_3v, y_train6, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train6_3v, batch_size = 64 )

    mse = MSE( y_train6, p )
    rmse = sqrt( mse )
    mae = MAE( y_train6, p )
    r2 = R2( y_train6, p )
    evs = EVS( y_train6, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1,rmse, mae, r2, evs))

C01 » RMSE: 3.9046, MAE: 2.1028, R2: 0.8217, EVS: 0.8223, 
C02 » RMSE: 3.8990, MAE: 2.1281, R2: 0.8222, EVS: 0.8229, 
C03 » RMSE: 3.8533, MAE: 2.1878, R2: 0.8263, EVS: 0.8295, 
C04 » RMSE: 3.8931, MAE: 2.1710, R2: 0.8227, EVS: 0.8245, 
C05 » RMSE: 3.7724, MAE: 2.0421, R2: 0.8336, EVS: 0.8338, 
C06 » RMSE: 3.9404, MAE: 1.9569, R2: 0.8184, EVS: 0.8192, 
C07 » RMSE: 3.9006, MAE: 2.1352, R2: 0.8220, EVS: 0.8224, 
C08 » RMSE: 3.8931, MAE: 2.1204, R2: 0.8227, EVS: 0.8233, 
C09 » RMSE: 3.9209, MAE: 2.1974, R2: 0.8202, EVS: 0.8223, 
C10 » RMSE: 3.9603, MAE: 2.2916, R2: 0.8166, EVS: 0.8212, 
C11 » RMSE: 3.8796, MAE: 2.0552, R2: 0.8240, EVS: 0.8246, 
C12 » RMSE: 3.7305, MAE: 2.0041, R2: 0.8372, EVS: 0.8375, 
C13 » RMSE: 3.7727, MAE: 2.1167, R2: 0.8335, EVS: 0.8359, 
C14 » RMSE: 3.9207, MAE: 2.0903, R2: 0.8202, EVS: 0.8206, 
C15 » RMSE: 3.8966, MAE: 2.1543, R2: 0.8224, EVS: 0.8236, 
C16 » RMSE: 3.9374, MAE: 2.1311, R2: 0.8187, EVS: 0.8195, 
C17 » RMSE: 3.9224, MAE: 2.1052, R2: 0.8201, EVS: 0.8210

In [7]:
# Train with 6 stations data, and then with 31. Compare them.
# First for the variables n,N for H/H0.

validation_data = ( x_dev_3v, y_dev )
for i in range(50):
    rads = radstimator(20, 15, 4)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-3vars-h/31stations-h-nNPhiH0 {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train_3v, y_train, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train_3v, batch_size = 64 )

    mse = MSE( y_train, p )
    rmse = sqrt( mse )
    mae = MAE( y_train, p )
    r2 = R2( y_train, p )
    evs = EVS( y_train, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1,rmse, mae, r2, evs))

C01 » RMSE: 3.0565, MAE: 1.7161, R2: 0.8746, EVS: 0.8746, 
C02 » RMSE: 3.0528, MAE: 1.6942, R2: 0.8749, EVS: 0.8749, 
C03 » RMSE: 3.0763, MAE: 1.7706, R2: 0.8729, EVS: 0.8736, 
C04 » RMSE: 3.0680, MAE: 1.7389, R2: 0.8736, EVS: 0.8737, 
C05 » RMSE: 3.0925, MAE: 1.8046, R2: 0.8716, EVS: 0.8731, 
C06 » RMSE: 3.0815, MAE: 1.7915, R2: 0.8725, EVS: 0.8741, 
C07 » RMSE: 3.0515, MAE: 1.7284, R2: 0.8750, EVS: 0.8750, 
C08 » RMSE: 3.1178, MAE: 1.8587, R2: 0.8695, EVS: 0.8714, 
C09 » RMSE: 3.0840, MAE: 1.7403, R2: 0.8723, EVS: 0.8728, 
C10 » RMSE: 3.1331, MAE: 1.7693, R2: 0.8682, EVS: 0.8686, 
C11 » RMSE: 3.0671, MAE: 1.7322, R2: 0.8737, EVS: 0.8743, 
C12 » RMSE: 3.0973, MAE: 1.7996, R2: 0.8712, EVS: 0.8734, 
C13 » RMSE: 3.0654, MAE: 1.7959, R2: 0.8738, EVS: 0.8745, 
C14 » RMSE: 3.0816, MAE: 1.8004, R2: 0.8725, EVS: 0.8746, 
C15 » RMSE: 3.0518, MAE: 1.7534, R2: 0.8749, EVS: 0.8753, 
C16 » RMSE: 3.0705, MAE: 1.7453, R2: 0.8734, EVS: 0.8734, 
C17 » RMSE: 3.0530, MAE: 1.6934, R2: 0.8748, EVS: 0.8749